<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</figure>

# Variables de entorno y `python-dotenv`
**Taller de Programación para el Análisis de Datos — FCE-UBA**

> *Cuando trabajamos con APIs o bases de datos necesitamos claves, usuarios y contraseñas. Si los escribimos dentro de la notebook, cualquiera que la reciba —un compañero, el docente, GitHub— se lleva también nuestras credenciales. Las variables de entorno y los archivos `.env` resuelven eso.*

---

**Contenido:**
1. El problema: secretos dentro del código
2. Variables de entorno — `os.environ` y `os.getenv`
3. El archivo `.env`
4. `python-dotenv` — `load_dotenv()`
5. Detalles importantes: todo es texto, `override`, `dotenv_values`
6. `.gitignore` y `.env.example`
7. Llevarlo a un módulo: `config.py`
8. Ejemplo real: consultar una API
9. ¿Y en Google Colab?
10. Errores comunes
11. Ejercicio integrador

---
## 1. El problema: secretos dentro del código

Así se ve una notebook típica que consulta una API:

```python
import requests

API_KEY = "Xk29fLm0QpZ7vB3nR8tY"        # ← la clave escrita en el código

url = f"https://api.polygon.io/v2/aggs/ticker/AAPL/prev?apiKey={API_KEY}"
r = requests.get(url)
```

Funciona, pero tiene varios problemas:

| Problema | Qué pasa |
|---|---|
| **Compartir la notebook** | Quien la recibe (un compañero, el docente) tiene tu clave |
| **Subirla a GitHub** | Hay bots que recorren GitHub buscando claves expuestas. En minutos pueden usarla |
| **Costos** | Muchas APIs (OpenAI, AWS, Google Cloud) cobran por uso: una clave filtrada puede generar gastos |
| **Cambiar la clave** | Hay que buscarla y reemplazarla en cada notebook donde la copiaste |
| **Trabajo en grupo** | Cada integrante tiene su propia clave, pero el código debería ser el mismo para todos |

La idea es **separar el código (que se comparte) de la configuración secreta (que no)**.

---
## 2. Variables de entorno — `os.environ` y `os.getenv`

El sistema operativo mantiene un diccionario de **variables de entorno**: pares `NOMBRE=valor` que cualquier programa puede leer.
Ya existen varias, por ejemplo `PATH` (dónde buscar programas) o `HOME`/`USERPROFILE` (tu carpeta de usuario).

En Python se acceden con el módulo `os`:

In [1]:
import os

# os.environ funciona como un diccionario
print(type(os.environ))
print("Cantidad de variables:", len(os.environ))

# Tu carpeta de usuario (HOME en Linux/Mac, USERPROFILE en Windows)
print("Carpeta de usuario:", os.getenv('HOME') or os.getenv('USERPROFILE'))

<class 'os._Environ'>
Cantidad de variables: 222
Carpeta de usuario: /root


In [ ]:
# ── Tres formas de leer una variable 

# 1. Como diccionario → si no existe, KeyError
try:
    os.environ['MI_CLAVE']
except KeyError:
    print("1. os.environ['MI_CLAVE']  → KeyError: no existe")

# 2. os.getenv → si no existe, devuelve None
print("2. os.getenv('MI_CLAVE')   →", os.getenv('MI_CLAVE'))

# 3. os.getenv con valor por defecto
print("3. con default             →", os.getenv('MI_CLAVE', 'valor_por_defecto'))

1. os.environ['MI_CLAVE']  → KeyError: no existe
2. os.getenv('MI_CLAVE')   → None
3. con default             → valor_por_defecto


In [3]:
# También se pueden crear desde Python (solo duran mientras el kernel esté vivo)
os.environ['MI_CLAVE'] = 'abc123'
print(os.getenv('MI_CLAVE'))

del os.environ['MI_CLAVE']
print(os.getenv('MI_CLAVE'))

abc123
None


Las variables de entorno se pueden definir desde la terminal antes de abrir Jupyter:

```bash
# Linux / Mac
export POLYGON_API_KEY=Xk29fLm0QpZ7vB3nR8tY

# Windows (PowerShell)
$env:POLYGON_API_KEY = "Xk29fLm0QpZ7vB3nR8tY"
```

Pero hay que hacerlo **cada vez** que se abre una terminal nueva, y con varias variables se vuelve tedioso. Para eso existe el archivo `.env`.

---
## 3. El archivo `.env`

Un `.env` es un archivo de texto plano, en la carpeta del proyecto, con una variable por línea:

```bash
# Comentarios con numeral
POLYGON_API_KEY=Xk29fLm0QpZ7vB3nR8tY

DB_HOST=localhost
DB_PORT=3306
DB_USER=marcelo
DB_PASSWORD="clave con espacios"

MODO_DEBUG=true
```

**Reglas del formato:**
- `NOMBRE=valor`, **sin espacios** alrededor del `=`
- Los nombres van en MAYÚSCULAS por convención
- Si el valor tiene espacios o `#`, va entre comillas
- El archivo se llama exactamente `.env` (empieza con punto, **sin** extensión `.txt`)

```
proyecto/
│
├── .env                ← secretos (NO se comparte)
├── .env.example        ← plantilla sin valores (SÍ se comparte)
├── .gitignore          ← le dice a git que ignore el .env
├── config.py           ← módulo que lee el .env
└── dotenv.ipynb        ← esta notebook
```

> ⚠️ En Windows, el Explorador a veces oculta los archivos que empiezan con punto y agrega `.txt` sin avisar. Conviene crearlo desde VS Code o desde Jupyter.

Para esta clase, creamos un `.env` de prueba desde la notebook (con valores **falsos**). Si ya tenés un `.env` en la carpeta, no se toca:

POLYGON_API_KEY=demo_9f8e7d6c5b4a3210
DB_HOST=localhost
DB_PORT=3306
DB_USER=alumno
DB_PASSWORD="clave con espacios #1"
DB_NAME=tpad

In [4]:
from pathlib import Path

contenido_demo = '''# .env de DEMOSTRACIÓN — valores falsos
POLYGON_API_KEY=demo_9f8e7d6c5b4a3210
DB_HOST=localhost
DB_PORT=3306
DB_USER=alumno
DB_PASSWORD="clave con espacios #1"
DB_NAME=tpad
MODO_DEBUG=true
'''

archivo = Path('.env')
if archivo.exists():
    print("Ya existe un .env — no se sobrescribe.")
else:
    archivo.write_text(contenido_demo, encoding='utf-8')
    print("Se creó .env de demostración.")

Se creó .env de demostración.


---
## 4. `python-dotenv` — `load_dotenv()`

`python-dotenv` es una librería que **lee el `.env` y carga sus variables en `os.environ`**. Después de eso, el código las lee con `os.getenv` como si se hubieran definido en el sistema.

Se instala una sola vez:

```bash
pip install python-dotenv
```

> El paquete se instala como `python-dotenv`, pero se importa como `dotenv`. Existe otro paquete llamado `dotenv` (sin `python-`) que **no** es este.

In [5]:
# Antes de cargar: la variable no existe
print("Antes :", os.getenv('POLYGON_API_KEY'))

from dotenv import load_dotenv

encontrado = load_dotenv()   # busca el .env y carga sus variables
print("¿Encontró el .env?", encontrado)

# Después de cargar
print("Después:", os.getenv('POLYGON_API_KEY'))

Antes : None
¿Encontró el .env? True
Después: demo_9f8e7d6c5b4a3210


**¿Dónde busca el `.env`?** Sin argumentos, `load_dotenv()` lo busca en la carpeta actual y, si no lo encuentra, va subiendo por las carpetas superiores. Para ver qué archivo encontró, o para indicar uno específico:

In [6]:
from dotenv import find_dotenv

print("Archivo encontrado:", find_dotenv())

# También se puede indicar la ruta exacta
# load_dotenv('ruta/a/mi/.env')
# load_dotenv(Path.home() / 'credenciales' / '.env')

Archivo encontrado: /home/claude/clase_dotenv/.env


**Nunca imprimas una clave completa**, ni siquiera para "ver si cargó". Las salidas de la notebook quedan guardadas en el `.ipynb` y viajan con el archivo. Mejor mostrar solo el final:

In [7]:
def ocultar(secreto, visibles=4):
    '''Muestra solo los últimos caracteres de un secreto.'''
    if not secreto:
        return '(vacío)'
    return '*' * max(len(secreto) - visibles, 4) + secreto[-visibles:]

clave = os.getenv('POLYGON_API_KEY')

print("✗ Mal :", clave)
print("✓ Bien:", ocultar(clave))
print("✓ Bien:", "cargada" if clave else "FALTA", f"({len(clave)} caracteres)")

✗ Mal : demo_9f8e7d6c5b4a3210
✓ Bien: *****************3210
✓ Bien: cargada (21 caracteres)


---
## 5. Detalles importantes

### 5.1 Todo llega como texto (`str`)

El `.env` es un archivo de texto: `DB_PORT=3306` no es el número 3306, es el string `'3306'`. Lo mismo con `true`/`false`.

In [8]:
puerto = os.getenv('DB_PORT')
debug  = os.getenv('MODO_DEBUG')

print(repr(puerto), type(puerto))
print(repr(debug),  type(debug))

# Hay que convertir explícitamente
puerto = int(puerto)
debug  = debug.lower() in ('true', '1', 'si', 'yes')

print(repr(puerto), type(puerto))
print(repr(debug),  type(debug))

# ⚠️ Trampa clásica: cualquier string no vacío es True
print("\nbool('false') =", bool('false'))

'3306' <class 'str'>
'true' <class 'str'>
3306 <class 'int'>
True <class 'bool'>

bool('false') = True


In [9]:
# Las comillas en el .env se quitan, y lo que va adentro se respeta
print(repr(os.getenv('DB_PASSWORD')))

'clave con espacios #1'


### 5.2 `override` — lo que ya está cargado gana

Por defecto, `load_dotenv()` **no pisa** variables que ya existen en `os.environ`.
Esto trae una consecuencia típica en Jupyter: si cambiás un valor en el `.env` y volvés a ejecutar `load_dotenv()`, **sigue el valor viejo**, porque ya estaba cargado en el kernel.

In [10]:
# Simulamos que la variable ya existía con otro valor
os.environ['MODO_DEBUG'] = 'false'

load_dotenv()                       # no pisa
print("Sin override :", os.getenv('MODO_DEBUG'))

load_dotenv(override=True)          # el .env manda
print("Con override :", os.getenv('MODO_DEBUG'))

Sin override : false
Con override : true


> **Regla práctica:** si modificaste el `.env` y no ves el cambio, usá `load_dotenv(override=True)` o reiniciá el kernel.

### 5.3 `dotenv_values` — leer sin tocar `os.environ`

Si solo querés ver el contenido del `.env` como diccionario, sin cargarlo en el entorno:

In [11]:
from dotenv import dotenv_values

valores = dotenv_values()   # dict con lo que hay en el .env

for nombre, valor in valores.items():
    es_secreto = any(p in nombre for p in ('KEY', 'PASSWORD', 'TOKEN', 'SECRET'))
    print(f"  {nombre:16} = {ocultar(valor) if es_secreto else valor}")

  POLYGON_API_KEY  = *****************3210
  DB_HOST          = localhost
  DB_PORT          = 3306
  DB_USER          = alumno
  DB_PASSWORD      = *****************s #1
  DB_NAME          = tpad
  MODO_DEBUG       = true


---
## 6. `.gitignore` y `.env.example`

Tener el `.env` no sirve de nada si después se sube a GitHub junto con el código. Dos archivos completan el esquema:

**`.gitignore`** — le indica a git qué archivos **no** debe subir nunca:

```bash
# Variables de entorno con secretos
.env
.env.*
!.env.example        # ← excepción: la plantilla sí se sube

# Python
__pycache__/
.ipynb_checkpoints/
```

**`.env.example`** — una copia del `.env` **con los nombres pero sin los valores**. Se sube al repositorio para que quien clone el proyecto sepa qué variables necesita:

```bash
POLYGON_API_KEY=
DB_HOST=localhost
DB_PORT=3306
DB_USER=
DB_PASSWORD=
DB_NAME=
MODO_DEBUG=false
```

El flujo para un compañero que descarga el proyecto:
1. Copia `.env.example` → `.env`
2. Completa con **sus** claves
3. Ejecuta la notebook sin cambiar una línea de código

| Archivo | ¿Tiene secretos? | ¿Se sube a GitHub? | ¿Se entrega en un TP? |
|---|---|---|---|
| `.env` | Sí | **No** | **No** |
| `.env.example` | No | Sí | Sí |
| `.gitignore` | No | Sí | — |
| `config.py` / notebooks | No | Sí | Sí |

> ⚠️ Si alguna vez subiste una clave por error, borrarla del archivo **no alcanza**: queda en el historial de git. Lo único seguro es **revocarla** en el sitio del proveedor y generar una nueva.

---
## 7. Llevarlo a un módulo: `config.py`

Juntando lo de la clase de módulos propios con esta: en vez de repetir `load_dotenv()` y las conversiones en cada notebook, armamos **un módulo** que lee la configuración una sola vez.

`config.py` (está en la carpeta):

```python
import os
from dotenv import load_dotenv

load_dotenv()

def _requerida(nombre):
    valor = os.getenv(nombre)
    if not valor:
        raise RuntimeError(f"Falta la variable de entorno '{nombre}' ...")
    return valor

POLYGON_API_KEY = _requerida('POLYGON_API_KEY')

DB = {
    'host':     os.getenv('DB_HOST', 'localhost'),
    'port':     int(os.getenv('DB_PORT', '3306')),
    ...
}

MODO_DEBUG = _booleana('MODO_DEBUG')
```

Ventajas:
- `load_dotenv()` y las conversiones de tipo están en **un solo lugar**
- Si falta una variable obligatoria, el error es **claro** y aparece al principio, no 30 celdas después con un `401 Unauthorized`
- Las notebooks solo hacen `from config import ...`

In [12]:
import config

print("API key :", config.ocultar(config.POLYGON_API_KEY))
print("Debug   :", config.MODO_DEBUG, type(config.MODO_DEBUG))
print("DB      :", {k: (config.ocultar(v) if k == 'password' else v) for k, v in config.DB.items()})

API key : *****************3210
Debug   : True <class 'bool'>
DB      : {'host': 'localhost', 'port': 3306, 'user': 'alumno', 'password': '*****************s #1', 'database': 'tpad'}


In [13]:
# ¿Qué pasa si falta una variable obligatoria?
# Simulamos el error con la misma función que usa config.py

try:
    config._requerida('OPENAI_API_KEY')
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError: Falta la variable de entorno 'OPENAI_API_KEY'.
Revisá que exista el archivo .env y que tenga OPENAI_API_KEY=...
(usá .env.example como plantilla)


---
## 8. Ejemplo real: consultar una API

Con todo armado, el código que consulta la API **no tiene ningún secreto** y es idéntico para todos los integrantes del grupo.

Además, siempre que la API lo permita, conviene mandar la clave en un **header** y no en la URL: las URLs quedan en logs, en el historial y en los mensajes de error.

> Esta celda necesita conexión y una clave real de [polygon.io](https://polygon.io) (el plan gratuito alcanza). Con la clave de demostración va a responder `401`.

In [ ]:
import requests
from config import POLYGON_API_KEY

ticker = 'AAPL'
url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/prev"

respuesta = requests.get(
    url,
    headers={'Authorization': f'Bearer {POLYGON_API_KEY}'},   # la clave va en el header
    timeout=10,
)

if respuesta.status_code == 200:
    datos = respuesta.json()['results'][0]
    print(f"{ticker} — cierre anterior: USD {datos['c']}  (volumen {datos['v']:,.0f})")
elif respuesta.status_code in (401, 403):
    print("Clave inválida o sin permisos. Revisá POLYGON_API_KEY en el .env")
else:
    print("Error", respuesta.status_code, respuesta.text[:200])

---
## 9. ¿Y en Google Colab?

En Colab no hay una carpeta del proyecto persistente, así que el `.env` se pierde al cerrar la sesión (y subirlo a Drive vuelve a exponerlo).
Colab tiene su propio mecanismo: **Secretos** (ícono de la llave 🔑 en la barra lateral izquierda).

1. Agregás `POLYGON_API_KEY` con su valor
2. Activás el acceso para la notebook
3. Lo leés así:

In [ ]:
from google.colab import userdata

POLYGON_API_KEY = userdata.get('POLYGON_API_KEY')

Los secretos quedan asociados a **tu cuenta**, no a la notebook: si la compartís, el otro no ve tu clave.

Para que una misma notebook funcione en las dos situaciones:

```python
try:
    from google.colab import userdata          # estamos en Colab
    POLYGON_API_KEY = userdata.get('POLYGON_API_KEY')
except ImportError:                            # estamos en Jupyter / VS Code
    from dotenv import load_dotenv
    load_dotenv()
    POLYGON_API_KEY = os.getenv('POLYGON_API_KEY')
```